In [ ]:
# CELL 1 - install (restart not needed). Settings: Accelerator = GPU T4 x2 (or P100), Internet = On
!pip install -q -U bitsandbytes peft accelerate dagshub mlflow

In [ ]:
# CELL 2 - config + data (upload train/val/test.jsonl as a Kaggle Dataset named "contextsql-data")
import os, json, glob, random, torch
BASE = "Qwen/Qwen2.5-Coder-3B-Instruct"
DAGSHUB_USER, DAGSHUB_REPO = "garvitwork", "contextsql"
MAX_LEN, EPOCHS, LR = 1024, 2, 2e-4
OUT = "/kaggle/working"

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found - add the contextsql-data dataset to this notebook"
    return hits[0]

load = lambda n: [json.loads(l) for l in open(find(n), encoding="utf-8")]
train, val, test = load("train.jsonl"), load("val.jsonl"), load("test.jsonl")
print(len(train), len(val), len(test))

In [ ]:
# CELL 3 - DagHub / MLflow login (add Kaggle secret DAGSHUB_USER_TOKEN via Add-ons > Secrets)
from kaggle_secrets import UserSecretsClient
import dagshub, dagshub.auth, mlflow
dagshub.auth.add_app_token(UserSecretsClient().get_secret("DAGSHUB_USER_TOKEN"))
dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
os.environ["MLFLOW_EXPERIMENT_NAME"] = "contextsql-finetune"

In [ ]:
# CELL 4 - model (4-bit) + LoRA + tokenization (loss only on the SQL answer)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE)
tok.padding_side = "right"
if tok.pad_token is None: tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0},
                                             torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def encode(r):
    p = tok(r["prompt"], add_special_tokens=False)["input_ids"]
    c = tok(r["completion"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    ids = (p + c)[:MAX_LEN]
    labels = ([-100] * len(p) + c)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

from datasets import Dataset
def prep(rows):
    ds = Dataset.from_list(rows).map(encode, remove_columns=list(rows[0].keys()))
    return ds.filter(lambda x: len(x["input_ids"]) < MAX_LEN)
train_ds, val_ds = prep(train), prep(val)
print("after length filter:", len(train_ds), len(val_ds))

In [ ]:
# CELL 5 - train (~30-60 min on T4)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
args = TrainingArguments(
    output_dir=f"{OUT}/ckpt", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4,
    fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,
    logging_steps=10, eval_strategy="steps", eval_steps=50, save_strategy="no",
    report_to="mlflow", gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
model.config.use_cache = False
trainer.train()
model.save_pretrained(f"{OUT}/adapter"); tok.save_pretrained(f"{OUT}/adapter")
print("adapter saved")

In [ ]:
# CELL 6 - generate test predictions: base model vs fine-tuned (same prompts)
model.config.use_cache = True
model.eval()
tok.padding_side = "left"

def clean(txt):
    txt = txt.strip().split("\n\n")[0]
    i = txt.find(";")
    return (txt[: i + 1] if i >= 0 else txt).strip()

@torch.no_grad()
def generate(rows, bs=8):
    outs = []
    for i in range(0, len(rows), bs):
        batch = [r["prompt"] for r in rows[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda:0")
        gen = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [clean(tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)) for g in gen]
    return outs

def dump(name, preds):
    with open(f"{OUT}/{name}", "w", encoding="utf-8") as f:
        for r, p in zip(test, preds):
            f.write(json.dumps({"source": r["source"], "template": r["template"],
                                "gold": r["completion"], "pred": p}) + "\n")

ft_preds = generate(test); dump("preds_finetuned.jsonl", ft_preds)
with model.disable_adapter():
    base_preds = generate(test); dump("preds_base.jsonl", base_preds)

with mlflow.start_run(run_name="test-predictions"):
    mlflow.log_artifact(f"{OUT}/preds_finetuned.jsonl"); mlflow.log_artifact(f"{OUT}/preds_base.jsonl")
print("done - download preds_*.jsonl and adapter/ from the Output tab")

In [ ]:
# CELL 7 (optional) - merge adapter + export GGUF for local use. Run after cells 1-6, then download the .gguf
import gc, subprocess
del model, trainer; gc.collect(); torch.cuda.empty_cache()
from peft import PeftModel
m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={"": "cpu"})
m = PeftModel.from_pretrained(m, f"{OUT}/adapter").merge_and_unload()
m.save_pretrained(f"{OUT}/merged"); tok.save_pretrained(f"{OUT}/merged")
del m; gc.collect()

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outfile /kaggle/working/contextsql-f16.gguf --outtype f16
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --target llama-quantize -j4 > /dev/null
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/working/contextsql-f16.gguf /kaggle/working/contextsql-q4_k_m.gguf Q4_K_M
!rm -rf /kaggle/working/merged /kaggle/working/contextsql-f16.gguf /kaggle/working/ckpt
!ls -lh /kaggle/working/*.gguf